# Module 4. `module3_sft_dataset.jsonl`로 SFTTrainer 기반 SFT 실습

이 노트북의 목표는 다음 4가지입니다.

1. `module3_sft_dataset.jsonl`을 Colab에서 불러옵니다.  
2. TRL의 `SFTTrainer`로 실제 SFT를 수행합니다.  
3. 튜닝 전후 출력을 비교합니다.  
4. 학습 결과와 관찰 메모를 저장합니다.

## 이번 모듈의 핵심
- 데이터 형식: `prompt-completion`
- 학습 방식: `SFTTrainer`
- 실습 전략: Colab 단일 GPU 기준 LoRA/PEFT 활용
- 비교 포인트: baseline 대비 말투, 형식, 간단한 task 수행 변화

## 준비물
- Module 3에서 만든 `module3_sft_dataset.jsonl`
- GPU가 활성화된 Colab 런타임 권장

## Step 1. 런타임 확인

먼저 현재 Colab 환경에서 Python, PyTorch, GPU 상태를 확인합니다.

### 확인 포인트
- GPU가 보이면 실습 속도가 훨씬 안정적입니다.
- GPU가 없더라도 코드는 실행 가능하지만 시간이 오래 걸릴 수 있습니다.

In [ ]:
import torch
import platform
import sys

print("Python version:", sys.version)
print("Platform:", platform.platform())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected. Colab 메뉴에서 Runtime > Change runtime type > GPU 설정을 권장합니다.")

## Step 2. 필수 라이브러리 설치

이번 실습에서 사용할 핵심 라이브러리를 설치합니다.

- `transformers`: 모델/토크나이저 로드
- `datasets`: JSONL 데이터셋 로드
- `trl`: `SFTTrainer`
- `peft`: LoRA/PEFT 설정
- `accelerate`: 학습 지원
- `sentencepiece`: tokenizer 지원

In [ ]:
!pip -q install -U transformers datasets trl peft accelerate sentencepiece

## Step 3. 라이브러리 import

학습과 저장, 비교를 위한 기본 라이브러리를 불러옵니다.

In [ ]:
import os
import json
from typing import List, Dict, Any

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

## Step 4. 실험 설정

이번 실습에서는 **같은 조건으로 전후 비교**하는 것이 중요합니다.

### 기본 설정
- 기본 모델: `HuggingFaceTB/SmolLM2-360M-Instruct`
- 입력 데이터: `module3_sft_dataset.jsonl`
- 출력 폴더: `/content/module4_sft_output`

GPU 여유가 있으면 모델을 1.7B로 바꿔도 됩니다.

In [ ]:
set_seed(42)

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
# MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

DEFAULT_DATA_PATH = "/content/module3_sft_dataset.jsonl"
OUTPUT_DIR = "/content/module4_sft_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("MODEL_NAME:", MODEL_NAME)
print("DEFAULT_DATA_PATH:", DEFAULT_DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

## Step 5. Module 3 데이터셋 준비

기본 경로에 `module3_sft_dataset.jsonl`이 있으면 그대로 사용합니다.  
파일이 없다면 직접 업로드할 수 있습니다.

### 기대 형식
각 줄은 아래와 같은 JSON 객체입니다.

```json
{"prompt": "27 * 14 의 결과만 답하세요.", "completion": "378"}
```

In [ ]:
DATA_PATH = DEFAULT_DATA_PATH

if not os.path.exists(DATA_PATH):
    print(f"File not found at {DATA_PATH}")
    print("업로드 창을 열어 module3_sft_dataset.jsonl 파일을 직접 업로드하세요.")
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise FileNotFoundError("업로드된 파일이 없습니다.")
    uploaded_name = list(uploaded.keys())[0]
    DATA_PATH = f"/content/{uploaded_name}"

print("Using DATA_PATH:", DATA_PATH)

## Step 6. 데이터셋 로드

로컬 JSONL 파일을 Hugging Face `datasets`로 로드합니다.

In [ ]:
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print(dataset)
print("First row:")
print(dataset[0])

## Step 7. 데이터셋 기본 검증

학습 전에 최소한의 형식 검증을 수행합니다.

### 체크 항목
- `prompt` 컬럼 존재 여부
- `completion` 컬럼 존재 여부
- 빈 문자열 샘플이 없는지

In [ ]:
required_columns = {"prompt", "completion"}
missing = required_columns - set(dataset.column_names)
if missing:
    raise ValueError(f"필수 컬럼이 없습니다: {missing}")

bad_rows = []
for i, row in enumerate(dataset):
    if not str(row["prompt"]).strip() or not str(row["completion"]).strip():
        bad_rows.append(i)

print("column_names:", dataset.column_names)
print("bad_rows:", bad_rows[:10], "... total:", len(bad_rows))
if bad_rows:
    raise ValueError("빈 prompt/completion 샘플이 존재합니다. 데이터셋을 정리해 주세요.")

## Step 8. train / eval 분리

작은 실습에서도 train / eval을 분리해 두면 과적합 여부를 감각적으로 보기 좋습니다.

In [ ]:
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("train size:", len(train_dataset))
print("eval size :", len(eval_dataset))

## Step 9. 샘플 확인

학습 전에 데이터가 의도한 스타일로 구성되어 있는지 직접 확인합니다.

In [ ]:
for i in range(min(3, len(train_dataset))):
    print("=" * 80)
    print("PROMPT:")
    print(train_dataset[i]["prompt"])
    print("\nCOMPLETION:")
    print(train_dataset[i]["completion"])

## Step 10. 토크나이저와 모델 로드

이번 실습은 `prompt-completion` 데이터를 이용한 SFT입니다.

### 참고 포인트
- `pad_token`이 없으면 `eos_token`으로 대체합니다.
- Colab에서는 전체 미세조정보다 LoRA 기반 학습이 더 실용적입니다.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype
)

model.config.use_cache = False

print("Loaded model and tokenizer.")
print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)

## Step 11. 생성 함수 정의

학습 전후 비교를 위해 동일한 생성 함수를 사용합니다.

In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=128):
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## Step 12. 학습 전 baseline 출력 확인

학습 전에 몇 개 prompt를 직접 넣어 baseline 출력을 확인합니다.

### 관찰 포인트
- 숫자만 답하라는 지시를 잘 따르는가
- JSON 형식을 지키는가
- 정중한 톤이 유지되는가

In [ ]:
test_prompts = [
    "27 * 14 의 결과만 답하세요.",
    "name, role 키를 가진 JSON만 출력하세요. name은 Alice, role은 engineer.",
    "고객 문의에 정중한 한국어로 3문장 이내로 답하세요: 배송이 늦어지고 있습니다."
]

before_outputs = []

print("=== BEFORE SFT ===")
for p in test_prompts:
    out = generate_text(model, tokenizer, p)
    before_outputs.append({"prompt": p, "output": out})
    print("=" * 80)
    print("PROMPT:", p)
    print("OUTPUT:", out)

## Step 13. LoRA 설정

Colab 환경에서는 LoRA/PEFT를 이용하면 훨씬 가볍게 SFT를 진행할 수 있습니다.

### 기본 전략
- 전체 모델을 전부 업데이트하지 않고 adapter만 학습
- 교육용 실습에서는 반복성과 비용 측면에서 유리

In [ ]:
peft_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear"
)

print(peft_config)

## Step 14. SFTConfig 설정

이번 실습은 `prompt-completion` 데이터셋을 사용하므로  
completion 부분에만 loss가 계산되도록 `completion_only_loss=True`를 명시합니다.

### 주요 설정
- `learning_rate=1e-4`: LoRA 실습용
- `gradient_accumulation_steps`: 작은 GPU에서 effective batch 확보
- `eval_strategy="epoch"`: epoch 단위 평가
- `save_strategy="epoch"`: epoch 종료 시 저장
- `packing=False`: 교육용으로 단순하게 유지

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=1e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    max_length=512,
    packing=False,
    completion_only_loss=True,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
)

print(training_args)

## Step 15. SFTTrainer 생성

이제 `train_dataset`, `eval_dataset`, `tokenizer`, `peft_config`를 이용해 trainer를 만듭니다.

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print(trainer)

## Step 16. 학습 실행

이 셀에서 실제 SFT를 수행합니다.

### 실습 팁
- 데이터셋이 아주 작으면 loss가 빠르게 내려갈 수 있습니다.
- 학습 시간이 길다면 `num_train_epochs`를 1 또는 2로 줄여도 됩니다.

In [ ]:
train_result = trainer.train()
print(train_result)

## Step 17. 모델 저장

학습이 끝나면 adapter와 tokenizer를 출력 폴더에 저장합니다.

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to:", OUTPUT_DIR)

## Step 18. 학습 후 출력 비교

같은 prompt를 다시 넣어 전후 차이를 비교합니다.

In [ ]:
trained_model = trainer.model

after_outputs = []

print("=== AFTER SFT ===")
for p in test_prompts:
    out = generate_text(trained_model, tokenizer, p)
    after_outputs.append({"prompt": p, "output": out})
    print("=" * 80)
    print("PROMPT:", p)
    print("OUTPUT:", out)

## Step 19. 전후 비교표 저장

학습 전후 출력을 한 파일에 저장해 두면 이후 평가 모듈에서 재사용하기 쉽습니다.

In [ ]:
comparison_rows = []
for before, after in zip(before_outputs, after_outputs):
    comparison_rows.append({
        "prompt": before["prompt"],
        "before": before["output"],
        "after": after["output"]
    })

comparison_path = os.path.join(OUTPUT_DIR, "module4_before_after_comparison.json")
with open(comparison_path, "w", encoding="utf-8") as f:
    json.dump(comparison_rows, f, ensure_ascii=False, indent=2)

print("Saved:", comparison_path)
comparison_rows

## Step 20. 간단한 관찰 메모 템플릿 저장

아래 템플릿을 채워서 이번 SFT 실습 관찰 메모를 남깁니다.

In [ ]:
notes = '''# Module 4 SFT Observation

## What changed after SFT?
- 

## Which category improved the most?
- 

## Which category still needs more work?
- 

## Is SFT enough for this task?
- 

## Candidate next step
- DPO / PPO / GRPO
- reason:
'''

notes_path = os.path.join(OUTPUT_DIR, "module4_sft_observation.md")
with open(notes_path, "w", encoding="utf-8") as f:
    f.write(notes)

print("Saved:", notes_path)
print(notes)

## Step 21. 결과 파일 확인

이번 모듈이 끝나면 아래 파일들이 생성되어야 합니다.

- 학습된 adapter / checkpoint
- `module4_before_after_comparison.json`
- `module4_sft_observation.md`

In [ ]:
print("Saved files:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    for name in files:
        print("-", os.path.join(root, name))

## Step 22. 선택 사항: 결과 다운로드

Colab 세션이 종료되기 전에 주요 결과 파일을 내려받을 수 있습니다.

In [ ]:
from google.colab import files

files.download(os.path.join(OUTPUT_DIR, "module4_before_after_comparison.json"))
files.download(os.path.join(OUTPUT_DIR, "module4_sft_observation.md"))

# Module 4 정리

이번 모듈에서는 Module 3에서 만든 `prompt-completion` 데이터셋을 사용해 실제 SFT를 수행했습니다.

## 이번 모듈에서 확인한 것
- `SFTTrainer`는 `prompt-completion` 형식을 직접 사용할 수 있습니다.
- completion-only loss 설정은 답변 생성 학습에 집중하게 해 줍니다.
- Colab에서는 LoRA 기반 SFT가 실용적입니다.
- SFT는 스타일, 형식, 기본 과제 수행을 먼저 안정화하는 데 적합합니다.

## 다음 단계로 연결
다음 Module에서는 아래 두 가지 방향이 자연스럽습니다.

1. **평가 모듈**
   - baseline vs SFT 비교
   - 정성/정량 루브릭 작성

2. **DPO 모듈**
   - `module3_dpo_dataset.jsonl` 사용
   - 선호쌍 기반 정렬 실습

즉, Module 4는 “정답 예시로 먼저 고치는 단계”라고 이해하면 됩니다.